# Import Required libraries

In [1]:
import os
import re
from typing import List, Dict
from tqdm import tqdm

# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


# EEG libraries
import mne

# EEG Data Processing

In [8]:
class EEGProcessor:
    def __init__(self, data_dir: str, sfreq: int = 160):
        """
        Initialize the EEGProcessor.
        
        Parameters:
            data_dir (str): Path to main data folder containing subject subfolders.
            sfreq (int): Sampling frequency of the EEG data.
        """
        self.data_dir = data_dir
        self.sfreq = sfreq
        self.metadata_df = None
        self.epochs_df = None

    # --- Trail classification ---
    @staticmethod
    def trail_classification(trail_num: int) -> str:
        if trail_num in [4, 6, 8, 10, 12, 14]:
            return "imaginary"
        elif trail_num in [1, 2]:
            return "baseline"
        else:
            return "motor"

    # --- Event type classification ---
    @staticmethod
    def event_type_classification(trail_num: int):
        if trail_num in [1, 2]:
            return ("rest", "NA", "NA")
        elif trail_num in [3, 4, 7, 8, 11, 12]:
            return ("rest", "left_fist", "right_fist")
        elif trail_num in [5, 6, 9, 10, 13, 14]:
            return ("rest", "both_fist", "both_feet")

    # --- Metadata extraction ---
    def extract_metadata(self) -> pd.DataFrame:
        metadata = []

        for subj in sorted(os.listdir(self.data_dir)):
            subj_path = os.path.join(self.data_dir, subj)
            if not os.path.isdir(subj_path):
                continue

            for file in sorted(os.listdir(subj_path)):
                if file.endswith(".edf") and not file.endswith(".edf.event"):
                    match = re.search(r"R(\d{2})\.edf$", file)
                    if match:
                        trial_num = int(match.group(1))
                        trail_type = self.trail_classification(trial_num)
                        event_type = self.event_type_classification(trial_num)

                        edf_file = os.path.join(subj_path, file)
                        event_file = edf_file + ".event" if os.path.exists(edf_file + ".event") else None

                        metadata.append({
                            "subject": subj,
                            "trial": trial_num,
                            "trail_type": trail_type,
                            "edf_file": edf_file,
                            "event_file": event_file,
                            "T0": event_type[0],
                            "T1": event_type[1],
                            "T2": event_type[2],
                        })

        self.metadata_df = pd.DataFrame(metadata)
        return self.metadata_df

    # --- Extract variable-duration epochs from a single file ---
    def get_epochs_from_file(self, file_name: str):
        raw = mne.io.read_raw_edf(file_name, preload=True)
        epochs = []

        for ann in raw.annotations:
            start = int(ann["onset"] * self.sfreq)
            stop = int((ann["onset"] + ann["duration"]) * self.sfreq)
            epoch_data = raw.get_data(start=start, stop=stop)
            label = ann["description"]
            epochs.append((epoch_data, label))

        return epochs

    # --- Extract all epochs for all metadata ---
    def extract_all_epochs(self) -> pd.DataFrame:
        if self.metadata_df is None:
            self.extract_metadata()

        all_epochs = []

        for _, row in self.metadata_df.iterrows():
            epochs = self.get_epochs_from_file(row['edf_file'])
            for epoch_data, label in epochs:
                all_epochs.append({
                    "subject": row['subject'],
                    "trial": row['trial'],
                    "trail_type": row['trail_type'],
                    "epoch_data": epoch_data,
                    "label": label, 
                    "label_description": row[label]
                })

        self.epochs_df = pd.DataFrame(all_epochs)
        return self.epochs_df


In [9]:
processor = EEGProcessor(data_dir="../data", sfreq=160)

# Extract metadata
metadata = processor.extract_metadata()
print(metadata.head())


# Extract all epochs
epochs_df = processor.extract_all_epochs()
print(epochs_df.head())


  subject  trial trail_type                  edf_file  \
0    S001      1   baseline  ../data/S001/S001R01.edf   
1    S001      2   baseline  ../data/S001/S001R02.edf   
2    S001      3      motor  ../data/S001/S001R03.edf   
3    S001      4  imaginary  ../data/S001/S001R04.edf   
4    S001      5      motor  ../data/S001/S001R05.edf   

                       event_file    T0         T1          T2  
0  ../data/S001/S001R01.edf.event  rest         NA          NA  
1  ../data/S001/S001R02.edf.event  rest         NA          NA  
2  ../data/S001/S001R03.edf.event  rest  left_fist  right_fist  
3  ../data/S001/S001R04.edf.event  rest  left_fist  right_fist  
4  ../data/S001/S001R05.edf.event  rest  both_fist   both_feet  
Extracting EDF parameters from /Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/PhysicoNet/data/S001/S001R01.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 9759  =

/var/folders/_p/pgtp_zhj7n3717r3m0prtkdh0000gn/T/ipykernel_7535/2275842751.py:71: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_name, preload=True)
/var/folders/_p/pgtp_zhj7n3717r3m0prtkdh0000gn/T/ipykernel_7535/2275842751.py:71: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_name, preload=True)
/var/folders/_p/pgtp_zhj7n3717r3m0prtkdh0000gn/T/ipykernel_7535/2275842751.py:71: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_name, preload=True)
/var/folders/_p/pgtp_zhj7n3717r3m0prtkdh0000gn/T/ipykernel_7535/2275842751.py:71: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_name, preload=True)
/var/folders/_p/pgtp_zhj7n3717r3m0prtkdh0000gn/T/ipykernel_7535/2275842751.py:71: RuntimeWarning: Limited 1 annotation(s) th

Reading 0 ... 15743  =      0.000 ...   122.992 secs...
Extracting EDF parameters from /Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/PhysicoNet/data/S101/S101R01.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 9759  =      0.000 ...    60.994 secs...
Extracting EDF parameters from /Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/PhysicoNet/data/S101/S101R02.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 9759  =      0.000 ...    60.994 secs...
Extracting EDF parameters from /Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/PhysicoNet/data/S101/S101R03.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /Users/ka

/var/folders/_p/pgtp_zhj7n3717r3m0prtkdh0000gn/T/ipykernel_7535/2275842751.py:71: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(file_name, preload=True)


Extracting EDF parameters from /Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/PhysicoNet/data/S102/S102R01.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 9759  =      0.000 ...    60.994 secs...
Extracting EDF parameters from /Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/PhysicoNet/data/S102/S102R02.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 9759  =      0.000 ...    60.994 secs...
Extracting EDF parameters from /Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/PhysicoNet/data/S102/S102R03.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Extracting EDF parameters from /Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisIn

In [10]:
# Access first epoch's data
print(epochs_df['epoch_data'][3].shape)
print(epochs_df['label_description'][3])

(64, 656)
right_fist


# Save the Dataframe

##### Sample Texting

In [2]:
raw = mne.io.read_raw_edf("../data/S001/S001R06.edf", preload=True)

raw.annotations

Extracting EDF parameters from /Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/PhysicoNet/data/S001/S001R06.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...


<Annotations | 30 segments: T0 (15), T1 (7), T2 (8)>

In [15]:
epochs_df[(epochs_df["trail_type"]=="imaginary") & (epochs_df["label_description"].isin(["both_fist", "both_feet"]))].groupby(["subject", "trial", "trail_type", "label_description","label"]).count()

epoch_data
subject trial trail_type label_description label            
S001    6     imaginary  both_feet         T2              8
                         both_fist         T1              7
        10    imaginary  both_feet         T2              8
                         both_fist         T1              7
        14    imaginary  both_feet         T2              8
...                                                      ...
S109    6     imaginary  both_fist         T1              8
        10    imaginary  both_feet         T2              7
                         both_fist         T1              8
        14    imaginary  both_feet         T2              8
                         both_fist         T1              7

[654 rows x 1 columns]

In [19]:
epochs_df[(epochs_df["trail_type"]=="imaginary") & (epochs_df["label_description"].isin(["both_fist", "both_feet"]))].to_pickle("../data/processed/eeg_processed.pkl")